# Build the "Same Place?" game dataset

Exports `game_pairs.json` for the address_matcher React app (`~/Downloads/address_matcher`).

Shape (decided 2026-11-15):
- `models`: `[{id, name, threshold}]`; threshold is on the probability, 0.5 for all.
- `pairs`: `[{pair_id, address_a, address_b, label, neg_type, probs: {model_id: p}}]`; the keys of `probs` are the foreign key into `models[].id`.
- No stored pass/fail: the app derives it as `(p >= threshold) == label`.
- Validation + test splits (no model trained on either; val was used to pick epochs and kept settings), `overture_case2s` dataset. Each pair carries its `split`. (Test only until 2026-11-21.)
- **Paper v2 (2026-11-21):** every model's probabilities come from the v2 release (`~/Downloads/address-matching`, case 1 fix, all models retrained): neural runs from its `predictions/`, XGBoost refitted on its feature cache. The local `checkpoints*/` folders here are v1 (43,844 test rows vs v2's 43,852) and are not used. ESIM = fastText (`esim_ft_mc1`, the v2 kept setting); the siamese BiLSTM (mean pooling) is the sixth model.

Boilerplate (setup, loading pairs, loading every model's test probabilities) is filled in; the TODO cells are David's.

In [1]:
# Setup: imports and paths
import os
import sys
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import json
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

from helpers.pairs import load_pairs, add_number_street
from helpers.features import get_processed_dfs

DATASET = "overture_case2s"
SEEDS = [0, 1, 2]
RELEASE = os.path.expanduser("~/Downloads/address-matching")   # paper v2 code release
FEATURE_CACHE = f"{RELEASE}/classical/features"   # v2 sentinel features; check_cache raises if the pairs differ
OUT_PATH = "game_pairs.json"

## 1. Pairs
All three splits are loaded because XGBoost has to be refitted on train; `val` and `test` go into the game.

In [2]:
# Load: case2s train / val / test pairs (same order the feature caches and saved probabilities were built from)
train, train_label = load_pairs(DATASET, "train")
val, val_label = load_pairs(DATASET, "val")
test, test_label = load_pairs(DATASET, "test")
train, val, test = [add_number_street(df) for df in [train, val, test]]

val = val.assign(label = val_label.values)
test = test.assign(label = test_label.values)
pd.concat([val.assign(split = "val"), test.assign(split = "test")]).groupby(["split", "neg_type"], observed = True).size().unstack()

neg_type,case0,case1,case2,case2s,case3
split,,,,,
test,4386,14379,15,1053,24019
val,4187,13732,34,894,23018


## 2. Model probabilities on val and test
One entry per model in `probs` (test) and `probs_val` (val): `{model_id: [p_seed0, p_seed1, p_seed2]}` (XGBoost is deterministic, so it gets one array).

In [3]:
# XGBoost (Step 1b): refit on the cached similarity features, same settings as run_against_models
train_X, val_X, test_X = get_processed_dfs([
    [train, f"{FEATURE_CACHE}/train.parquet"],
    [val, f"{FEATURE_CACHE}/val.parquet"],
    [test, f"{FEATURE_CACHE}/test.parquet"],
])
xgb = XGBClassifier(random_state = 0).fit(train_X, train_label)

probs = {"xgb": [xgb.predict_proba(test_X)[:, 1]]}
probs_val = {"xgb": [xgb.predict_proba(val_X)[:, 1]]}
probs

{'xgb': [array([0.9929986 , 0.9999294 , 0.9999294 , ..., 0.5656129 , 0.00589   ,
         0.00141572], dtype=float32)]}

In [4]:
# Neural models (Steps 3, 4, 5a, 5b): v2 saved sigmoid outputs, one file per seed, rows in load_pairs order
SAVED = {
    "siamese": f"{RELEASE}/predictions/siamese_mean",
    "esim":    f"{RELEASE}/predictions/esim_ft_mc1",
    "bi":      f"{RELEASE}/predictions/distilbert_bi",
    "cross":   f"{RELEASE}/predictions/distilbert_cross",
}
for model_id, folder in SAVED.items():
    probs[model_id] = [np.load(f"{folder}/probs_test_seed_{s}.npy").ravel() for s in SEEDS]
    probs_val[model_id] = [np.load(f"{folder}/probs_val_seed_{s}.npy").ravel() for s in SEEDS]

{m: [len(p) for p in ps] for m, ps in probs.items()}

{'xgb': [43852],
 'siamese': [43852, 43852, 43852],
 'esim': [43852, 43852, 43852],
 'bi': [43852, 43852, 43852],
 'cross': [43852, 43852, 43852]}

In [5]:
# Check: the saved probabilities line up with these test rows, against the paper (v2, Table 15)
# (1) right length; (2) per seed, % of non-identical matches found and % of look-alike streets (case2s) rejected
# on the FULL test split should fall inside the paper's min–max (XGBoost: 83.6 / 87.0; siamese 87.0–89.3 / 69.8–73.1;
# ESIM fastText 87.3–90.3 / 73.5–80.4; bi 89.0–90.9 / 57.6–68.8; cross 80.3–83.0 / 86.1–88.8).
for split_probs, df in [(probs, test), (probs_val, val)]:
    for model_id, ps in split_probs.items():
        for s, p in zip(SEEDS, ps):
            assert len(p) == len(df), (model_id, s, len(p), len(df))

non_identical = (test.neg_type == "case0") & (test.address_a != test.address_b)
look_alike = test.neg_type == "case2s"
pd.DataFrame({
    f"{m} seed {s}": {"non-identical found": round(100 * (p[non_identical] >= 0.5).mean(), 1),
                      "look-alike rejected": round(100 * (p[look_alike] < 0.5).mean(), 1)}
    for m, ps in probs.items() for s, p in zip(SEEDS, ps)
}).T


,non-identical found,look-alike rejected
xgb seed 0,83.6,87.0
siamese seed 0,87.0,73.0
siamese seed 1,87.2,73.1
siamese seed 2,89.3,69.8
esim seed 0,88.6,74.2
esim seed 1,90.3,73.5
esim seed 2,87.3,80.4
bi seed 0,89.0,66.1
bi seed 1,90.9,57.6
bi seed 2,89.9,68.8


## 3. Game pool (TODO, David)
Filter `val` and `test` (separately) to the pairs the game shows: non-identical positives, `case2`, `case2s` (+ any easy tier). Optional: cap rows per unordered street pair.

In [6]:
# Pool per split: non-identical positives, case2, case2s; at most 2 rows per (street_a, street_b) within a split.
# `row` = the pair's position in its split's load_pairs order, the key into that split's saved probabilities.
def game_pool(df):
    pool = df[df["neg_type"].isin(["case2", "case2s"]) | ((df["neg_type"] == "case0") & (df["address_a"] != df["address_b"]))]
    return pool.groupby(["street_a", "street_b"]).head(2).reset_index().rename(columns = {"index": "row"})

game_pool_df = pd.concat([game_pool(val).assign(split = "val"), game_pool(test).assign(split = "test")], ignore_index = True)
# A few pairs exist twice in the dataset: once as a real look-alike (case2) and once as a street swap (case2s) with the
# same ids (2 in val, 2 in test). The game keys pairs by id, so keep one row each, the real case2 one (see Diagnostics D3).
game_pool_df = (game_pool_df.sort_values("neg_type", key = lambda s: s == "case2s", kind = "stable")
                .drop_duplicates(["id_a", "id_b"]).sort_index())
game_pool_df


,row,id_a,id_b,address_a,address_b,zip_a,zip_b,split,neg_type,number_a,street_a,number_b,street_b,both_number_missing,number_missing,label
0,1,141ccfbc-b855-4946-a228-eb61a3d291b0,141ccfbc-b855-4946-a228-eb61a3d291b0,229 SEYMOUR RD,SEYMOUR RD,06525,06525,val,case0,229,SEYMOUR RD,NaN,SEYMOUR RD,False,True,1
1,12,7869f618-da2f-4ee2-838f-3c79e03fb31b,7869f618-da2f-4ee2-838f-3c79e03fb31b,RACEBROOK RD,1027 RACEBROOK RD,06525,06525,val,case0,NaN,RACEBROOK RD,1027,RACEBROOK RD,False,True,1
2,15,70794f96-436e-4fb2-9d8c-92bcf6e12351,70794f96-436e-4fb2-9d8c-92bcf6e12351,990 RACEBROOK ROAD,990 RACEBROOK RD,06525,06525,val,case0,990,RACEBROOK ROAD,990,RACEBROOK RD,False,False,1
3,47,f409e39c-3914-472c-9067-7e7a4622e2bb,f409e39c-3914-472c-9067-7e7a4622e2bb,9 COTS ST 2C,9 COTS ST,06484,06484,val,case0,9,COTS ST 2C,9,COTS ST,False,False,1
4,48,a2904989-ac35-4e51-a931-10dd853b6f24,a2904989-ac35-4e51-a931-10dd853b6f24,MEDICAL CENTER 9 COTS ST,9 COTS ST,06484,06484,val,case0,NaN,MEDICAL CENTER 9 COTS ST,9,COTS ST,False,True,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2614,43845,b6cd4b72-4f1e-47db-ae98-f41255fb94d1,475f47f5-1b9f-4953-bc48-dbfd537c7ced,59 PARK ST,59 PEARL ST,06082,06082,test,case2s,59,PARK ST,59,PEARL ST,False,False,0
2615,43846,9b035885-e2f9-43ce-b0e1-7329c4ba95ea,8757e690-a77f-4d9d-9a4a-6a4e4d13ca5d,90 OLIVER ST,90 RIVER ST,06606,06606,test,case2s,90,OLIVER ST,90,RIVER ST,False,False,0
2616,43847,65fe314b-ee42-45b0-877c-204ac3a7964c,56281496-26fd-4c68-b306-efe03d3b6b1f,701 COTTAGE GROVE RD BLDG B,701 COTTAGE GROVE RD STE 103,06002,06002,test,case2s,701,COTTAGE GROVE RD BLDG B,701,COTTAGE GROVE RD STE 103,False,False,0
2617,43848,f3033a97-e995-463d-ad49-d0c3b03498c7,113969c0-4509-47a6-a0f3-5728bce510f2,1 SHAKER HILL RD,1 SHAKER RD,06082,06082,test,case2s,1,SHAKER HILL RD,1,SHAKER RD,False,False,0


In [7]:
game_pool_df.groupby(["split", "neg_type"]).size().unstack()


neg_type,case0,case2,case2s
split,,,
test,663,14,725
val,619,32,563


## 4. One probability per model (TODO, David)
Decide: one seed, or the mean of the three seeds' probabilities.

In [8]:
# Mean of the seeds' probabilities per model (XGBoost has one), per split, keyed by (split, row)
def mean_probs(split_probs, n):
    ret = pd.DataFrame({f"{m}_logit": np.mean(ps, axis = 0) for m, ps in split_probs.items()})
    assert len(ret) == n
    return ret

scores = pd.concat([mean_probs(probs_val, len(val)).assign(split = "val"),
                    mean_probs(probs, len(test)).assign(split = "test")]).rename_axis("row").reset_index()


In [9]:
merged_df = game_pool_df.merge(scores, on = ["split", "row"], how = "left", validate = "one_to_one").drop(columns = "row")
merged_df


,id_a,id_b,address_a,address_b,zip_a,zip_b,split,neg_type,number_a,street_a,number_b,street_b,both_number_missing,number_missing,label,xgb_logit,siamese_logit,esim_logit,bi_logit,cross_logit
0,141ccfbc-b855-4946-a228-eb61a3d291b0,141ccfbc-b855-4946-a228-eb61a3d291b0,229 SEYMOUR RD,SEYMOUR RD,06525,06525,val,case0,229,SEYMOUR RD,NaN,SEYMOUR RD,False,True,1,0.990829,0.982177,0.972032,0.996561,0.996728
1,7869f618-da2f-4ee2-838f-3c79e03fb31b,7869f618-da2f-4ee2-838f-3c79e03fb31b,RACEBROOK RD,1027 RACEBROOK RD,06525,06525,val,case0,NaN,RACEBROOK RD,1027,RACEBROOK RD,False,True,1,0.990829,0.999562,0.682086,0.974443,0.002012
2,70794f96-436e-4fb2-9d8c-92bcf6e12351,70794f96-436e-4fb2-9d8c-92bcf6e12351,990 RACEBROOK ROAD,990 RACEBROOK RD,06525,06525,val,case0,990,RACEBROOK ROAD,990,RACEBROOK RD,False,False,1,0.981060,0.996318,0.996490,0.999999,0.995827
3,f409e39c-3914-472c-9067-7e7a4622e2bb,f409e39c-3914-472c-9067-7e7a4622e2bb,9 COTS ST 2C,9 COTS ST,06484,06484,val,case0,9,COTS ST 2C,9,COTS ST,False,False,1,0.929647,0.905208,0.980222,0.999952,0.996750
4,a2904989-ac35-4e51-a931-10dd853b6f24,a2904989-ac35-4e51-a931-10dd853b6f24,MEDICAL CENTER 9 COTS ST,9 COTS ST,06484,06484,val,case0,NaN,MEDICAL CENTER 9 COTS ST,9,COTS ST,False,True,1,0.994099,0.792228,0.989187,0.999999,0.996792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2611,b6cd4b72-4f1e-47db-ae98-f41255fb94d1,475f47f5-1b9f-4953-bc48-dbfd537c7ced,59 PARK ST,59 PEARL ST,06082,06082,test,case2s,59,PARK ST,59,PEARL ST,False,False,0,0.003519,0.064617,0.020459,0.000056,0.001285
2612,9b035885-e2f9-43ce-b0e1-7329c4ba95ea,8757e690-a77f-4d9d-9a4a-6a4e4d13ca5d,90 OLIVER ST,90 RIVER ST,06606,06606,test,case2s,90,OLIVER ST,90,RIVER ST,False,False,0,0.053822,0.213857,0.596214,0.003463,0.001555
2613,65fe314b-ee42-45b0-877c-204ac3a7964c,56281496-26fd-4c68-b306-efe03d3b6b1f,701 COTTAGE GROVE RD BLDG B,701 COTTAGE GROVE RD STE 103,06002,06002,test,case2s,701,COTTAGE GROVE RD BLDG B,701,COTTAGE GROVE RD STE 103,False,False,0,0.062048,0.294749,0.049877,0.938749,0.052190
2614,f3033a97-e995-463d-ad49-d0c3b03498c7,113969c0-4509-47a6-a0f3-5728bce510f2,1 SHAKER HILL RD,1 SHAKER RD,06082,06082,test,case2s,1,SHAKER HILL RD,1,SHAKER RD,False,False,0,0.858963,0.724875,0.259566,0.999800,0.586990


## 4b. LLM (zero-shot, OpenAI)
One call per pool pair. The model answers YES / NO **first**, then one sentence of reasoning.
- Probability: from the first token's log-probabilities, `p = p_yes / (p_yes + p_no)`.
- Reasoning: the rest of the text. It is written after the answer, so it is a justification, not the process that produced it.

Answers are cached to `LLM_CACHE` (keyed by the two address strings), so a crash or re-run never pays twice and the scores stay fixed.
Key: `OPENAI_API_KEY` in `ml-geocoding/.env` (git-ignored), never in this notebook.

In [10]:
# LLM setup: key, client, prompt, cache
import re
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../../.env")           # ml-geocoding/.env -> OPENAI_API_KEY
LLM_MODEL = "gpt-4o-mini"           # check OpenAI's current cheapest model that returns logprobs
LLM_CACHE = "llm_cache.parquet"
PROMPT = """Do these two address records refer to the same location?
A: {a}
B: {b}
Answer YES or NO first, then one short sentence explaining why."""

client = OpenAI()                   # reads OPENAI_API_KEY from the environment

CACHE_COLS = ["address_a", "address_b", "lp_yes", "lp_no", "reasoning"]
cache = pd.read_parquet(LLM_CACHE) if os.path.exists(LLM_CACHE) else pd.DataFrame(columns = CACHE_COLS)
assert list(cache.columns) == CACHE_COLS, "old cache format: delete llm_cache.parquet and re-run"
done = set(zip(cache.address_a, cache.address_b))
len(done)

2585

In [11]:
# One pair -> (log-prob of YES, log-prob of NO, reasoning sentence)
# The log-probs come from the FIRST token only (None if not in its top 5); the reasoning is the text after YES/NO.
def score_pair(a, b):
    resp = client.chat.completions.create(
        model = LLM_MODEL,
        messages = [{"role": "user", "content": PROMPT.format(a = a, b = b)}],
        temperature = 0,
        max_tokens = 60,
        logprobs = True,
        top_logprobs = 5,
    )
    choice = resp.choices[0]
    lp = {"YES": None, "NO": None}
    for t in choice.logprobs.content[0].top_logprobs:   # "YES", "Yes", " yes" all count as YES; keep the most likely spelling
        key = t.token.strip().upper()
        if key in lp and lp[key] is None:
            lp[key] = t.logprob
    reasoning = re.sub(r"^\s*(YES|NO)\b[\s.,:;!-]*", "", choice.message.content or "", flags = re.I).strip()
    return lp["YES"], lp["NO"], reasoning

# score_pair("41 VILLAGE LN", "41 VILLAGE LANE")     # smoke test: one paid call (off: the cache covers every pool pair)

In [12]:
# Score every pool pair not yet in the cache; save every 50 calls
todo = [(a, b) for a, b in zip(merged_df.address_a, merged_df.address_b) if (a, b) not in done]
CALL_API = True   # 2026-11-21: val pairs added to the pool (David approved the ~1,216 calls); False = only list the misses
if not CALL_API:
    print(f"{len(todo)} pool pairs missing from {LLM_CACHE}:", todo[:20])
    todo = []
rows = []
for i, (a, b) in enumerate(todo):
    lp_yes, lp_no, reasoning = score_pair(a, b)
    rows.append({"address_a": a, "address_b": b, "lp_yes": lp_yes, "lp_no": lp_no, "reasoning": reasoning})
    if (i + 1) % 50 == 0 or i + 1 == len(todo):
        cache = pd.concat([cache, pd.DataFrame(rows)], ignore_index = True)
        cache.to_parquet(LLM_CACHE, index = False)
        done |= {(r["address_a"], r["address_b"]) for r in rows}
        rows = []
        print(f"{i + 1} / {len(todo)}")

In [13]:
# Log-probabilities -> one probability per pair; join probability + reasoning onto merged_df
# Missing token = probability 0. Both missing -> NaN (the section 5 check will catch it).
p_yes = np.exp(cache.lp_yes.astype(float)).fillna(0)
p_no = np.exp(cache.lp_no.astype(float)).fillna(0)
cache["llm_logit"] = (p_yes / (p_yes + p_no)).where(p_yes + p_no > 0)   # it's a probability; named like the other columns
cache["llm_reasoning"] = cache["reasoning"]

merged_df = merged_df.drop(columns = ["llm_logit", "llm_reasoning"], errors = "ignore").merge(
    cache[["address_a", "address_b", "llm_logit", "llm_reasoning"]].drop_duplicates(["address_a", "address_b"]),
    on = ["address_a", "address_b"], how = "left")
merged_df[["llm_logit", "llm_reasoning"]].head()

,llm_logit,llm_reasoning
0,0.000028,because the first address specifies a street n...
1,0.999993,because both addresses refer to the same stree...
2,1.000000,"both addresses refer to the same location as ""..."
3,0.997527,because both addresses share the same street n...
4,0.060087,because the first address specifies a medical ...


## 5. Build + check + export (TODO, David)
Build `{"models": [...], "pairs": [...]}`, assert every pair's `probs` keys == the set of `models[].id` and every p is in [0, 1], then `json.dump` to `OUT_PATH`.

In [14]:
MODELS = [{"id": "xgb",     "name": "XGBoost",                       "threshold": 0.5},
          {"id": "siamese", "name": "Siamese BiLSTM",                "threshold": 0.5},
          {"id": "esim",    "name": "ESIM (fastText)",               "threshold": 0.5},
          {"id": "bi",    "name": "Bi-encoder",      "threshold": 0.5},
          {"id": "cross", "name": "Cross-encoder",   "threshold": 0.5},
          {"id": "llm",   "name": "LLM (zero-shot)", "threshold": 0.5}]
ids = [m["id"] for m in MODELS]

# llm's probability sits in probs["llm"] like every other model; its reasoning is a pair-level field
pairs = [{"pair_id": f"{r.id_a}_{r.id_b}",
          "address_a": r.address_a, "address_b": r.address_b,
          "label": int(r.label), "neg_type": r.neg_type, "split": r.split,
          "probs": {m: round(float(getattr(r, f"{m}_logit")), 4) for m in ids},
          "llm_reasoning": r.llm_reasoning}
         for r in merged_df.itertuples()]

assert len({p["pair_id"] for p in pairs}) == len(pairs)                       # pair_id is unique across val + test
assert all(set(p["probs"]) == set(ids) for p in pairs)                       # foreign key
assert all(0 <= v <= 1 for p in pairs for v in p["probs"].values())          # also catches NaN (e.g. an unscored LLM pair)
assert all(isinstance(p["llm_reasoning"], str) for p in pairs)               # every pair has reasoning

with open(OUT_PATH, "w") as f:
    json.dump({"models": MODELS, "pairs": pairs}, f)
len(pairs)

2616

In [15]:
merged_df.head()

,id_a,id_b,address_a,address_b,zip_a,zip_b,split,neg_type,number_a,street_a,...,both_number_missing,number_missing,label,xgb_logit,siamese_logit,esim_logit,bi_logit,cross_logit,llm_logit,llm_reasoning
0,141ccfbc-b855-4946-a228-eb61a3d291b0,141ccfbc-b855-4946-a228-eb61a3d291b0,229 SEYMOUR RD,SEYMOUR RD,06525,06525,val,case0,229,SEYMOUR RD,...,False,True,1,0.990829,0.982177,0.972032,0.996561,0.996728,0.000028,because the first address specifies a street n...
1,7869f618-da2f-4ee2-838f-3c79e03fb31b,7869f618-da2f-4ee2-838f-3c79e03fb31b,RACEBROOK RD,1027 RACEBROOK RD,06525,06525,val,case0,NaN,RACEBROOK RD,...,False,True,1,0.990829,0.999562,0.682086,0.974443,0.002012,0.999993,because both addresses refer to the same stree...
2,70794f96-436e-4fb2-9d8c-92bcf6e12351,70794f96-436e-4fb2-9d8c-92bcf6e12351,990 RACEBROOK ROAD,990 RACEBROOK RD,06525,06525,val,case0,990,RACEBROOK ROAD,...,False,False,1,0.981060,0.996318,0.996490,0.999999,0.995827,1.000000,"both addresses refer to the same location as ""..."
3,f409e39c-3914-472c-9067-7e7a4622e2bb,f409e39c-3914-472c-9067-7e7a4622e2bb,9 COTS ST 2C,9 COTS ST,06484,06484,val,case0,9,COTS ST 2C,...,False,False,1,0.929647,0.905208,0.980222,0.999952,0.996750,0.997527,because both addresses share the same street n...
4,a2904989-ac35-4e51-a931-10dd853b6f24,a2904989-ac35-4e51-a931-10dd853b6f24,MEDICAL CENTER 9 COTS ST,9 COTS ST,06484,06484,val,case0,NaN,MEDICAL CENTER 9 COTS ST,...,False,True,1,0.994099,0.792228,0.989187,0.999999,0.996792,0.060087,because the first address specifies a medical ...


In [16]:
model_logits = ["siamese_logit", "esim_logit", "bi_logit", "cross_logit", "xgb_logit", "llm_logit"]
n = len(merged_df)
for col in model_logits:
    print(f"{'='*20} {col} {'='*20}")
    positives = ((merged_df[col] >= 0.5).astype(int) ==  merged_df["label"]).sum()
    print(f"matched: {positives} || missed: {n - positives}")

==================== siamese_logit ====================
matched: 2112 || missed: 504
==================== esim_logit ====================
matched: 2130 || missed: 486
==================== bi_logit ====================
matched: 1979 || missed: 637
==================== cross_logit ====================
matched: 2225 || missed: 391
==================== xgb_logit ====================
matched: 2252 || missed: 364
==================== llm_logit ====================
matched: 1976 || missed: 640


## Diagnostics
One-off queries and their results (question → result → query).

**D1 (2026-11-21, v2 re-export). Do the exported pool's rates sit close to the paper's Table 15?**
Result: yes, with the expected offset from the pool cap (2 rows per street pair keeps 663 of 843 non-identical matches and 727 of 1,053 look-alikes). 3-seed means on the pool, % non-identical found / % look-alike rejected: XGBoost 81.1 / 88.9 (paper 83.6 / 87.0), siamese 85.5 / 74.7 (87.0–89.3 / 69.8–73.1), ESIM fastText 85.8 / 78.4 (87.3–90.3 / 73.5–80.4), bi 87.8 / 64.2 (89.0–90.9 / 57.6–68.8), cross 79.3 / 89.8 (80.3–83.0 / 86.1–88.8), LLM 70.1 / 79.4 (not in the paper). Same ordering as the paper. On the full test split, every seed reproduces Table 15 exactly (cell under "Check: the saved probabilities line up").
Query:
```python
pos, la = merged_df.neg_type == "case0", merged_df.neg_type == "case2s"
pd.DataFrame({m: {"found": 100 * (merged_df.loc[pos, f"{m}_logit"] >= .5).mean(),
                  "rejected": 100 * (merged_df.loc[la, f"{m}_logit"] < .5).mean()} for m in ids}).T.round(1)
```

**D2 (2026-11-21). How does difficulty spread with 6 models?**
Result: models wrong 0–6 → 681 / 288 / 182 / 108 / 52 / 70 / 23 pairs, so easy 681, medium 470, hard 230, absurd 23 (v1 with 5 models: 665 / 465 / 190 / 84). Absurd shrinks because all six now have to miss.
Query:
```python
wrong = sum(((merged_df[f"{m}_logit"] >= .5) != (merged_df.label == 1)).astype(int) for m in ids)
wrong.value_counts().sort_index()
```

**D3 (2026-11-21). Why did the pair_id uniqueness check fail after adding val?**
Result: 4 pairs (2 val, 2 test) are in the dataset twice with the same ids and label, once as `case2` and once as `case2s`, e.g. 35 BUNKER HILL RD / 35 CANDEE HILL RD (val) and 1 PEERLESS WAY / 1 PEARSON WAY (test), each in both orders. The test-only export already had the 2 test ones; the app kept one silently. Now the pool keeps the `case2` row. Data note for the write-up: these rows are counted in both the case2 and case2s slices of the paper's tables.
Query:
```python
pid = game_pool_df.id_a + "_" + game_pool_df.id_b
game_pool_df[pid.duplicated(keep = False)][["split", "neg_type", "address_a", "address_b"]]
```
